In [1]:
import json
import numpy as np
import torch
from torch.optim import AdamW
from torch.utils.data import ConcatDataset, DataLoader
import os
import sys
from pathlib import Path
import random
from transformers import AutoTokenizer, AutoModelForTokenClassification
from sklearn.model_selection import KFold

# set path to project root and import custom classes and functions
project_root = Path.cwd() / "../../../"
sys.path.append(str(project_root.resolve()))

from utils.classification import TokenDataset, collate_bert_ner, train_bert
from utils.evaluation import run_testset_ner

# set seed and specify the device
seed = 0
device = torch.device("mps" if torch.mps.is_available() else "cpu")
print(f"Code running on: {device}")

Code running on: mps


In [2]:
with open(project_root / "01_data/classification/training_validation_sets/ner/training_set.json", "r") as f:
    data = json.load(f)

# initialize tag dictionary
tag_dict = {"O"}

# loop through all sentences
for task in data:
    if task["annotations"]:
        for annotation in task["annotations"]:
            label = annotation["tag"][0:2]
            tag_dict.add(f"B-{label}")
            tag_dict.add(f"I-{label}")

# sort the tag dictionary
tag_list = sorted(tag_dict)

# dictionaries that convert from id to tag and vice versa
tag_to_id = {tag: i for i, tag in enumerate(tag_list)}
id_to_tag = {id: label for label, id in tag_to_id.items()}

# load the results from hyperparameter tuning
with open(project_root / "04_classification_models/group_mention_detection/bert_models/hyperparameter_tuning_results/ht_bert_ner.json", "r") as f:
    hyperparameter_tuning_results = json.load(f)

# set all model names
model_name = "roberta-base"

In [3]:
def ids2words(sentence, tokenizer, max_len=128):
    encoding = tokenizer(
        sentence,
        return_offsets_mapping=True,
        truncation=True,
        max_length=max_len
    )
    word_ids = encoding.word_ids()
    offsets = encoding["offset_mapping"]

    word_to_chars = {}

    for token_idx, word_id in enumerate(word_ids):
        if word_id is None:
            continue

        start_char, end_char = offsets[token_idx]

        if word_id not in word_to_chars:
            word_to_chars[word_id] = [start_char, end_char]
        else:
            word_to_chars[word_id][1] = end_char

    word_dict = {}
    for word_id, (start, end) in word_to_chars.items():
        word_dict[word_id] = sentence[start:end]

    return word_dict

def span2text(span_word_ids, word_dict):
    sorted_ids = sorted(span_word_ids)
    return " ".join(word_dict[i] for i in sorted_ids)

def summarize(values):
    values = np.array(values)
    n = len(values)
    mean = np.mean(values)
    sd = np.std(values)
    ci = 1.96 * (sd/np.sqrt(n))
    return {
        "mean": mean,
        "sd": sd,
        "lower": mean - ci,
        "upper": mean + ci
        }

In [4]:
# get the optimal hyperparameters for this model
epochs = hyperparameter_tuning_results[model_name]["best_epoch"]
lr = hyperparameter_tuning_results[model_name]["best_params"]["lr"]
batch_size = hyperparameter_tuning_results[model_name]["best_params"]["batch_size"]
weight_decay = hyperparameter_tuning_results[model_name]["best_params"]["weight_decay"]

# define the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# create cross-validation object for 5 folds
k = 5
kf = KFold(n_splits=k, shuffle=True, random_state=seed)

# store share of error types per fold
fps_shares_folds = []
fns_shares_folds = []
boundary_mismatches_shares_folds = []

# store the actual errors in a list
fps_mentions = []
fns_mentions = []
boundary_mismatches_mentions = []

# randomly split the training data into 5 folds and loop over them
for fold, (train_idx, val_idx) in enumerate(kf.split(data)):

    print(f"\nFold number: {fold + 1}")

    # create the training and validation fold based on the provided indices
    train_fold_data = [data[i] for i in train_idx]
    val_fold_data = [data[i] for i in val_idx]

    # create tensor dataset and respective data loaders
    augmented_train_data_generative = []
    num_augmentations_to_add = int(len(train_fold_data) * 0.25)
    candidates_for_augmentation = random.sample(
         train_fold_data, k=min(num_augmentations_to_add, len(train_fold_data))
         )
    for original_item in candidates_for_augmentation:
        if "augmentations" in original_item and len(original_item["augmentations"]) > 0:
            aug_gen = original_item["augmentations"][-1]
            augmented_train_data_generative.append({
                "id": f"{original_item['id']}_aug_{aug_gen['method']}",
                "sentence": aug_gen["sentence"],
                "annotations": aug_gen["annotations"]
                })
    train_fold_data_gen_augmentations = train_fold_data + augmented_train_data_generative

    train_dataset = TokenDataset(train_fold_data_gen_augmentations, tokenizer, tag_to_id, max_len=128)
    val_dataset = TokenDataset(val_fold_data, tokenizer, tag_to_id, max_len=128)
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_bert_ner)
    val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_bert_ner)

    # create the model and optimizer
    model = AutoModelForTokenClassification.from_pretrained(
        model_name,
        num_labels=len(tag_to_id),
        id2label=id_to_tag,
        label2id=tag_to_id
        ).to(device)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    # train the model with the optimal configuration
    train_bert(train_dataloader, model, optimizer, epochs, device, which_task="ner")

    # run test fold through the trained model
    all_true_spans, all_pred_spans, _ = run_testset_ner(
        model=model, test_dataloader=val_dataloader, id2tag=id_to_tag, device=device, for_metric="cross_span"
        )
    
    # counting variables per fold
    tps = 0
    fps = 0
    fns = 0
    boundary_mismatches = 0

    # loop through all sentences
    for sentence_idx, (sentence_true, sentence_preds) in enumerate(zip(all_true_spans, all_pred_spans)):

        # get the sentence and word ids
        sentence = val_fold_data[sentence_idx]["sentence"]
        ids_to_words_dict = ids2words(sentence, tokenizer)
        
        # get all unique word ids for each span as a set
        true_sets = [set(gt) for gt in sentence_true]
        pred_sets = [set(p) for p in sentence_preds]

        # empty set that stores all visited true word ids
        matched_true_idx = set()

        # loop through all predicted spans
        for p_set in pred_sets:
            best_overlap = 0
            best_idx = None
            # loop through true spans
            for i, t_set in enumerate(true_sets):
                # check overlap and store if it is a new best
                overlap = len(p_set & t_set)
                if overlap > best_overlap:
                    best_overlap = overlap
                    best_idx = i
            
            # if there was a match, calculate metrics for this predicted span
            if best_overlap > 0:
                t_set = true_sets[best_idx]
                precision = best_overlap / len(p_set)
                recall = best_overlap / len(t_set)
                matched_true_idx.add(best_idx)

                if (precision == 1.0) & (recall == 1.0):
                    tps += 1
                else:
                    boundary_mismatches += 1
                    boundary_mismatches_mentions.append(
                        {"original_sentence": sentence,
                         "gold_span": span2text(t_set, ids_to_words_dict),
                         "pred_span": span2text(p_set, ids_to_words_dict)}
                    )
            
            # if no match, this is a false positive
            else:
                fps += 1
                fps_mentions.append(
                    {"original_sentence": sentence,
                     "pred_span": span2text(p_set, ids_to_words_dict)}
                     )
            
        # loop through all true spans 
        for i, t_set in enumerate(true_sets):
            # if not already visited, this is a false negative
            if i not in matched_true_idx:
                fns += 1
                fns_mentions.append(
                    {"original_sentence": sentence,
                     "gold_span": span2text(t_set, ids_to_words_dict)}
                )

    # calculate shares and append to lists
    total_errors = fps + fns + boundary_mismatches
    fps_shares_folds.append(fps/total_errors)
    fns_shares_folds.append(fns/total_errors)
    boundary_mismatches_shares_folds.append(boundary_mismatches/total_errors)

# compute averages and export
mean_error_shares = {
    "false_positives": summarize(fps_shares_folds),
    "false_negatives": summarize(fns_shares_folds),
    "boundary_mismatches": summarize(boundary_mismatches_shares_folds)
}
with open(project_root / "04_classification_models/group_mention_detection/bert_models/error_analysis_results/cv_error_shares.json", "w") as f:
    json.dump(mean_error_shares, f, indent=4)

# export all error mentions
all_error_mentions = {
    "false_positives": fps_mentions,
    "false_negatives": fns_mentions,
    "boundary_mismatches": boundary_mismatches_mentions
}
with open(project_root / "04_classification_models/group_mention_detection/bert_models/error_analysis_results/all_error_mentions.json", "w") as f:
    json.dump(all_error_mentions, f, indent=4)


Fold number: 1


Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/8


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Training: 100%|██████████| 250/250 [01:48<00:00,  2.31it/s, loss=0.0372]


Average training loss: 0.1148
Epoch 2/8


...

Average training loss: 0.0058
Epoch 7/8


Training: 100%|██████████| 250/250 [01:44<00:00,  2.40it/s, loss=0.0127]  


Average training loss: 0.0049
Epoch 8/8


Training: 100%|██████████| 250/250 [01:44<00:00,  2.40it/s, loss=0.00703] 


Average training loss: 0.0025


In [5]:
with open(project_root / "04_classification_models/group_mention_detection/bert_models/error_analysis_results/cv_error_shares.json", "r") as f:
    mean_error_shares = json.load(f)

with open(project_root / "04_classification_models/group_mention_detection/bert_models/error_analysis_results/all_error_mentions.json", "r") as f:
    all_error_mentions = json.load(f)

fps_mentions = all_error_mentions["false_positives"]
fns_mentions = all_error_mentions["false_negatives"]
boundary_mismatches_mentions = all_error_mentions["boundary_mismatches"]

In [7]:
# show the error proportions
print(mean_error_shares)

{'false_positives': {'mean': 0.2469307644904462, 'sd': 0.10693692572061075, 'lower': 0.15319641628572273, 'upper': 0.3406651126951697}, 'false_negatives': {'mean': 0.19734813962930672, 'sd': 0.09924995861303969, 'lower': 0.11035171517397012, 'upper': 0.2843445640846433}, 'boundary_mismatches': {'mean': 0.555721095880247, 'sd': 0.06238623305241314, 'lower': 0.5010371515578337, 'upper': 0.6104050402026603}}


In [8]:
# randomly sample 100 mentions from each category
n_fp = min(100, len(fps_mentions))
n_fn = min(100, len(fns_mentions))
n_boundary = min(100, len(boundary_mismatches_mentions))

sample_fp = random.sample(fps_mentions, n_fp)
sample_fn = random.sample(fns_mentions, n_fn)
sample_boundary = random.sample(boundary_mismatches_mentions, n_boundary)